Diagrama UML

![Diagrama UML](docs/diagrama_uml.png)

DDL – DATA DEFINITION LANGUAGE

In [ ]:
import sqlite3

# Función para inicializar la base de datos y crear las tablas necesarias
def inicializar_db(libros_db = "libros.db"):
    conexion = sqlite3.connect(libros_db) # Cambia "libros.db" por la ruta deseada para tu base de datos
    cursor = conexion.cursor() # Crear las tablas necesarias para almacenar la información de los libros, autores y categorías

    cursor.executescript('''
        CREATE TABLE IF NOT EXISTS categorias (
            id_categoria INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre TEXT UNIQUE NOT NULL
        );
    ''') # Crear la tabla de categorías con un campo de ID autoincremental y un campo de nombre único
    print("Tabla Categorias creada.")

    cursor.executescript('''    
        CREATE TABLE IF NOT EXISTS autores (
            id_autor INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre_autor TEXT UNIQUE NOT NULL
        );
        ''') # Crear la tabla de autores con un campo de ID autoincremental y un campo de nombre único
    print("Tabla Autores creada.")
           
    cursor.executescript(''' 
        CREATE TABLE IF NOT EXISTS libros (
            id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            precio REAL,
            rating INTEGER,
            stock INTEGER,
            link TEXT,
            categoria_id INTEGER,
            FOREIGN KEY (categoria_id) REFERENCES categorias (id_categoria)
        );
    ''') # Crear la tabla de libros con un campo de ID autoincremental, campos para el título, precio, rating, stock, link y una clave foránea que referencia a la tabla de categorías
    print("Tabla Libros creada.")

    cursor.executescript('''        
        CREATE TABLE IF NOT EXISTS libro_autor (
            libro_id INTEGER,
            autor_id INTEGER,
            FOREIGN KEY (libro_id) REFERENCES libros (id_libro),
            FOREIGN KEY (autor_id) REFERENCES autores (id_autor),
            PRIMARY KEY (libro_id, autor_id)
        );                                      
    ''') # Crear la tabla de relación entre libros y autores con claves foráneas que referencian a las tablas de libros y autores, y una clave primaria compuesta por libro_id y autor_id
    print("Tabla Libro_Autor creada.")

    conexion.commit() # Guardar los cambios en la base de datos
    return conexion # Devolver la conexión a la base de datos para su uso posterior

conn = inicializar_db() # Llamar a la función para inicializar la base de datos y almacenar la conexión en la variable conn

Tabla Categorias creada.
Tabla Autores creada.
Tabla Libros creada.
Tabla Libro_Autor creada.


WEB SCRAPING

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import random
from urllib.parse import urljoin, quote_plus
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import re
from dotenv import load_dotenv

load_dotenv() # Carga las variables de entorno desde el archivo .env

URL_BASE = "https://books.toscrape.com/" # URL base del sitio web que vamos a scrapear
API_KEY = os.getenv("GOOGLE_API_KEY") # Obtener la API KEY de Google Books desde las variables de entorno

if not API_KEY: # Si no se encuentra la API KEY, mostramos una advertencia
    print("⚠️ ADVERTENCIA: No se encontró la API KEY. Configura tu archivo .env")

ESTRELLAS = {
    'One': 1, 
    'Two': 2, 
    'Three': 3, 
    'Four': 4, 
    'Five': 5
} # Diccionario para convertir el texto de rating en número de estrellas

# Función para crear una sesión de requests con un User-Agent personalizado
def crear_sesion(): 
    session = requests.Session() # Crear una sesión para mantener las cookies y configuraciones entre peticiones
    session.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'}) # Actualizar los headers de la sesión para incluir un User-Agent común y evitar bloqueos por parte del sitio web
    return session # Devolver la sesión creada para su uso en las peticiones

# Función para obtener el contenido HTML de una página y parsearlo con BeautifulSoup
def obtener_soup(url, session):
    try: # Intentamos hacer la petición a la URL utilizando la sesión proporcionada
        respuesta = session.get(url) # Hacemos la petición GET a la URL utilizando la sesión para mantener las configuraciones y cookies
        respuesta.raise_for_status() # Verificamos que la respuesta sea exitosa (código 200), si no, se lanzará una excepción
        soup = BeautifulSoup(respuesta.text, 'lxml') # Parseamos el contenido HTML de la respuesta utilizando BeautifulSoup con el parser 'lxml' para facilitar la extracción de datos
        return soup # Devolver el objeto BeautifulSoup con el contenido de la página
    
    except Exception as error: # Si ocurre cualquier error durante la petición o el parseo, lo capturamos y mostramos un mensaje de error
        print(f"Error en {url}: {error}")
        return None # Devolver None en caso de error para indicar que no se pudo obtener el contenido de la página

# Función para extraer el stock disponible a partir del texto de disponibilidad    
def extraer_stock(disponibilidad_texto):
    match = re.search(r'\((\d+) available\)', disponibilidad_texto) # Utilizamos una expresión regular para buscar el número de unidades disponibles en el texto de disponibilidad, buscando un patrón como "(20 available)"
    if match:
        return int(match.group(1)) # Si encontramos una coincidencia, convertimos el número encontrado a entero y lo devolvemos como stock disponible
    return 0

# Función para extraer los datos de un libro a partir del elemento HTML que lo representa, la categoría a la que pertenece, la URL actual y la sesión de requests    
def extraer_datos_libro(libros, categoria_nombre, url_actual, session):
    titulo = libros.find('h3').a['title'] # Extraemos el título del libro buscando el elemento 'h3' dentro del artículo del libro, luego accedemos al enlace 'a' dentro de 'h3' y obtenemos el atributo 'title' que contiene el título del libro
    precio_text = libros.find("p", class_="price_color").text # Extraemos el precio del libro buscando el elemento 'p' con la clase 'price_color' dentro del artículo del libro y obteniendo su texto, que contiene el precio con el símbolo de moneda
    precio_simb = precio_text.replace("£", "").replace('Â', '').strip() # Limpiamos el texto del precio eliminando el símbolo de libra (£) y cualquier carácter extraño como 'Â', y luego eliminamos espacios en blanco adicionales utilizando strip()
    precio = float(precio_simb) # Convertimos el precio limpio a un número decimal (float) para poder realizar cálculos o comparaciones posteriormente
    rating_objeto = libros.find("p", class_="star-rating").get("class") # Extraemos el rating del libro buscando el elemento 'p' con la clase 'star-rating' dentro del artículo del libro y obteniendo su atributo 'class', que contiene una lista de clases donde una de ellas indica el nivel de rating (por ejemplo, "One", "Two", etc.)
    rating_text = rating_objeto[1] # El segundo elemento de la lista de clases es el que indica el texto del rating, lo almacenamos en la variable rating_text
    rating = ESTRELLAS.get(rating_text, 0) # Utilizamos el diccionario ESTRELLAS para convertir el texto del rating en un número de estrellas, si el texto no se encuentra en el diccionario, devolvemos 0 como valor predeterminado

    enlace_relativo = libros.find('h3').a['href'] # Extraemos el enlace relativo al detalle del libro buscando el elemento 'h3' dentro del artículo del libro, luego accedemos al enlace 'a' dentro de 'h3' y obtenemos el atributo 'href' que contiene la URL relativa al detalle del libro
    link_libro = urljoin(url_actual, enlace_relativo) # Construimos la URL completa del detalle del libro utilizando urljoin para combinar la URL actual de la categoría con el enlace relativo extraído, lo que nos da la URL completa para acceder al detalle del libro   

    try:
        respuesta_detalle = session.get(link_libro, timeout=10) # Hacemos una petición GET a la URL del detalle del libro utilizando la sesión para mantener las configuraciones y cookies, y establecemos un tiempo de espera de 10 segundos para evitar bloqueos o esperas prolongadas en caso de problemas con la conexión o el sitio web

        if respuesta_detalle.status_code == 200: # Verificamos que la respuesta del detalle del libro sea exitosa (código 200) antes de intentar parsear el contenido, si no es exitosa, asumimos que no hay stock disponible y asignamos 0
            soup_detalle = BeautifulSoup(respuesta_detalle.text, 'lxml') # Parseamos el contenido HTML del detalle del libro utilizando BeautifulSoup con el parser 'lxml' para facilitar la extracción de datos
            texto_disponibilidad = soup_detalle.find('p', class_="instock availability").text.strip() # Buscamos el elemento 'p' con la clase 'instock availability' dentro del detalle del libro, que contiene el texto de disponibilidad, y obtenemos su texto limpio utilizando strip() para eliminar espacios en blanco adicionales
            stock = extraer_stock(texto_disponibilidad) # Llamamos a la función extraer_stock pasando el texto de disponibilidad para obtener el número de unidades disponibles en stock, que se devuelve como un entero

        else:
            stock = 0 # Si la respuesta del detalle del libro no es exitosa, asignamos 0 al stock para indicar que no hay unidades disponibles o que no se pudo obtener la información de stock
    except Exception as e: # Si ocurre cualquier error durante la petición o el parseo del detalle del libro, lo capturamos y mostramos un mensaje de error indicando que hubo un problema al buscar el stock para ese libro específico, y asignamos 0 al stock como valor predeterminado en caso de error
        print(f"Error buscando stock en {titulo}: {e}")
        stock = 0

    return {
        "titulo": titulo,
        "autor": "Pendiente",
        "precio": precio,
        "categoria": categoria_nombre,
        "rating": rating,
        "stock": stock,
        "link": link_libro
    } # Devolvemos un diccionario con los datos extraídos del libro, incluyendo el título, un campo de autor que inicialmente se establece como "Pendiente" para ser actualizado posteriormente, el precio, la categoría a la que pertenece, el rating en número de estrellas, el stock disponible y el enlace al detalle del libro

# Función para obtener los enlaces de todas las categorías disponibles en el sitio web, utilizando la sesión de requests proporcionada para hacer las peticiones y parsear el contenido
def obtener_links_categorias(session):
    soup = obtener_soup(URL_BASE, session) # Obtenemos el contenido de la página principal del sitio web utilizando la función obtener_soup y la sesión de requests proporcionada, lo que nos devuelve un objeto BeautifulSoup con el contenido HTML de la página principal
    lista_categoria = [] # Creamos una lista vacía para almacenar los datos de las categorías que vamos a extraer

    panel_lateral = soup.find("div", class_="side_categories") # Buscamos el elemento 'div' con la clase 'side_categories' dentro del contenido de la página principal, que es donde se encuentran los enlaces a las diferentes categorías de libros, y almacenamos ese elemento en la variable panel_lateral para poder extraer los enlaces de las categorías desde allí
    enlaces = panel_lateral.find_all("a") # Dentro del panel lateral de categorías, buscamos todos los elementos 'a' que representan los enlaces a las categorías, y almacenamos esa lista de elementos en la variable enlaces para poder iterar sobre ellos y extraer la información de cada categoría

    for enlace in enlaces[1:]: # Iteramos sobre la lista de enlaces a categorías, comenzando desde el segundo elemento (índice 1) para omitir el primer enlace que suele ser "All products" o similar, y procesamos cada enlace para extraer el nombre de la categoría y la URL completa
        nombre_cat = enlace.text.strip() # Extraemos el texto del enlace, que contiene el nombre de la categoría, y utilizamos strip() para eliminar espacios en blanco adicionales, almacenando el resultado en la variable nombre_cat
        ruta_relativa = enlace["href"] # Extraemos el atributo 'href' del enlace, que contiene la ruta relativa a la página de la categoría, y almacenamos esa ruta en la variable ruta_relativa
        link_completo = URL_BASE + ruta_relativa # Construimos la URL completa de la categoría concatenando la URL base del sitio web con la ruta relativa extraída del enlace, lo que nos da la URL completa para acceder a la página de esa categoría específica

        lista_categoria.append({
            "nombre": nombre_cat,
            "url": link_completo
        }) # Agregamos un diccionario con el nombre de la categoría y su URL completa a la lista de categorías, lo que nos permite tener una estructura organizada con la información de cada categoría para su posterior procesamiento

    return lista_categoria # Devolvemos la lista de categorías extraídas, donde cada categoría es representada como un diccionario con su nombre y URL completa

# Función para obtener el autor de un libro utilizando la API de Google Books, con una lógica que permite realizar consultas a la API solo en un porcentaje de los casos para evitar bloqueos, y en caso de no obtener resultados o no tener una API KEY, asignar un autor simulado
def obtener_autor(libro):
    titulo = libro['titulo'] # Extraemos el título del libro del diccionario que se le pasa a la función, para usarlo como consulta en la API de Google Books y buscar el autor correspondiente a ese título
    
    if API_KEY and random.random() < 0.30: # 30% de las veces consultamos
        try: 
            titulo_codificado = quote_plus(titulo) # Codificamos el título del libro para que pueda ser utilizado en la URL de la API de Google Books, utilizando quote_plus para manejar espacios y caracteres especiales correctamente en la consulta
            url = f"https://www.googleapis.com/books/v1/volumes?q=intitle:{titulo_codificado}&key={API_KEY}&fields=items(volumeInfo/authors)" # Construimos la URL de la API de Google Books para buscar libros por título, incluyendo la API KEY y especificando que solo queremos obtener el campo de autores en la respuesta para optimizar la consulta
            resp = requests.get(url, timeout=3) # Hacemos una petición GET a la URL de la API de Google Books con un tiempo de espera de 3 segundos para evitar bloqueos o esperas prolongadas en caso de problemas con la conexión o la API, y almacenamos la respuesta en la variable resp
            
            if resp.status_code == 200: # Verificamos que la respuesta de la API sea exitosa (código 200) antes de intentar procesar los datos, si no es exitosa, asumimos que no se pudo obtener el autor y caemos al fallback del autor simulado
                data = resp.json() # Convertimos la respuesta de la API de Google Books a formato JSON para poder acceder a los datos de manera estructurada, y almacenamos ese JSON en la variable data
                # Google devuelve una lista 'items', verificamos si existe
                if "items" in data:
                    info = data["items"][0]["volumeInfo"] # Accedemos al primer resultado de la lista 'items' y luego al campo 'volumeInfo' que contiene la información del libro, almacenando esa información en la variable info para poder extraer el autor
                    # Verificamos si tiene autores
                    if "authors" in info:
                        autor_real = info["authors"][0] # Tomamos el primero de los autores listados en la respuesta de la API, ya que puede haber varios autores, y almacenamos ese autor en la variable autor_real para asignarlo al libro
                        libro['autor'] = autor_real # Actualizamos el campo 'autor' del diccionario del libro con el nombre del autor real obtenido de la API de Google Books, reemplazando el valor "Pendiente" que teníamos inicialmente
                        return f"✅ Google: {titulo[:15]}... -> {autor_real}" # Devolvemos un mensaje indicando que se obtuvo el autor real desde Google, mostrando una parte del título del libro y el nombre del autor para confirmar que se realizó la consulta correctamente
            
        except Exception as e:
            pass # Si falla, usamos el simulado
            
        # Google permite más peticiones, pero mantenemos una pausa corta
        time.sleep(0.5)
    
    # Fallback: Si no hay key, falló la búsqueda o cayó en el 70% restante
    libro['autor'] = "Autor Simulado"
    return f"🤖 Simulado: {titulo[:15]}..."

# Función para procesar una categoría completa de libros, descargando los datos de cada libro en esa categoría y manejando la paginación para obtener todos los libros disponibles, utilizando la sesión de requests proporcionada para hacer las peticiones y parsear el contenido
def procesar_categoria(cat, session):
    nombre = cat["nombre"] # Extraemos el nombre de la categoría del diccionario que se le pasa a la función, para usarlo como parte de los datos de cada libro que se extraiga de esa categoría y para mostrar información de progreso durante el procesamiento
    url_actual = cat["url"] # Extraemos la URL de la categoría del diccionario que se le pasa a la función, para usarla como punto de partida para hacer las peticiones y obtener el contenido de la página de esa categoría, y luego manejar la paginación para obtener todas las páginas de esa categoría si es necesario
    libros_de_esta_cat = [] # Creamos una lista vacía para almacenar los datos de los libros que se extraigan de esta categoría, lo que nos permite organizar los datos por categoría y luego unirlos con los datos de otras categorías si es necesario
    
    while True: # Iniciamos un bucle infinito para manejar la paginación de la categoría, lo que nos permite seguir obteniendo libros de esa categoría mientras haya páginas disponibles, y salimos del bucle cuando ya no haya más páginas que procesar
        soup = obtener_soup(url_actual, session) # Obtenemos el contenido de la página actual de la categoría utilizando la función obtener_soup y la sesión de requests proporcionada, lo que nos devuelve un objeto BeautifulSoup con el contenido HTML de esa página, o None si hubo un error al obtener el contenido
        if not soup:
            break # Si no pudimos obtener el contenido de la página (soup es None), salimos del bucle para evitar errores al intentar procesar una página que no se pudo cargar correctamente

        articulos = soup.find_all("article", class_="product_pod") # Buscamos todos los elementos 'article' con la clase 'product_pod' dentro del contenido de la página de la categoría, que representan cada uno un libro, y almacenamos esa lista de elementos en la variable articulos para poder iterar sobre ellos y extraer los datos de cada libro utilizando la función extraer_datos_libro
        for art in articulos: # Iteramos sobre la lista de artículos que representan los libros en la página de la categoría, y para cada artículo, llamamos a la función extraer_datos_libro pasando el artículo, el nombre de la categoría, la URL actual y la sesión de requests para obtener un diccionario con los datos del libro, que luego agregamos a la lista de libros de esta categoría
            datos = extraer_datos_libro(art, nombre, url_actual, session) # Llamamos a la función extraer_datos_libro pasando el artículo del libro, el nombre de la categoría, la URL actual y la sesión de requests para obtener un diccionario con los datos del libro, incluyendo título, autor (inicialmente "Pendiente"), precio, categoría, rating, stock y enlace al detalle del libro
            libros_de_esta_cat.append(datos) # Agregamos el diccionario con los datos del libro a la lista de libros de esta categoría, lo que nos permite tener una colección organizada de los libros que pertenecen a esta categoría específica

        boton_next = soup.find("li", class_="next") # Buscamos el elemento 'li' con la clase 'next' dentro del contenido de la página de la categoría, que representa el botón o enlace para ir a la siguiente página de esa categoría, y almacenamos ese elemento en la variable boton_next para verificar si existe y manejar la paginación correctamente
        if boton_next:
            enlace_next = boton_next.a['href'] # Si encontramos el botón de "next", extraemos el enlace relativo a la siguiente página accediendo al elemento 'a' dentro del botón y obteniendo su atributo 'href', que contiene la URL relativa a la siguiente página de la categoría
            url_actual = urljoin(url_actual, enlace_next) # Construimos la URL completa de la siguiente página utilizando urljoin para combinar la URL actual de la categoría con el enlace relativo extraído del botón de "next", lo que nos da la URL completa para acceder a la siguiente página de esa categoría y continuar el proceso de extracción de libros en esa nueva página
        else:
            break
            
    # Imprimimos el resultado al finalizar la categoría
    print(f"✅ Categoría procesada: {nombre:.<30} {len(libros_de_esta_cat)} libros")
    return libros_de_esta_cat # Devolvemos la lista de libros extraídos de esta categoría, donde cada libro es representado como un diccionario con sus datos correspondientes

# Función principal para ejecutar el proceso completo de scraping, que incluye la descarga masiva de libros de todas las categorías y la actualización de los autores utilizando la API de Google Books, manejando la concurrencia con ThreadPoolExecutor para optimizar el tiempo de ejecución
def ejecutar_scraping():
    # --- FASE 1: SCRAPING DEL SITIO (Descarga masiva) ---
    print("🚀 FASE 1: Descargando libros de todas las categorías...")
    inicio = time.time() # Guardamos el tiempo de inicio para medir el tiempo que tarda esta fase del proceso
    
    mi_sesion = crear_sesion() # Creamos una sesión de requests utilizando la función crear_sesion, lo que nos permite mantener las configuraciones y cookies entre las peticiones que vamos a hacer durante el proceso de scraping, y almacenamos esa sesión en la variable mi_sesion para usarla en las funciones que hacen las peticiones al sitio web
    categorias = obtener_links_categorias(mi_sesion) # Obtenemos la lista de categorías disponibles en el sitio web utilizando la función obtener_links_categorias y pasando la sesión de requests que creamos, lo que nos devuelve una lista de diccionarios donde cada diccionario representa una categoría con su nombre y URL completa, y almacenamos esa lista en la variable categorias para procesar cada categoría posteriormente
    
    lista_libros = [] # Creamos una lista vacía para almacenar los datos de todos los libros que vamos a extraer de todas las categorías, lo que nos permite tener una colección completa de los libros obtenidos durante el proceso de scraping para su posterior uso o análisis
    
    # Usamos ThreadPoolExecutor para procesar cada categoría en paralelo, lo que nos permite optimizar el tiempo de ejecución al aprovechar la concurrencia para descargar los libros de varias categorías al mismo tiempo, y max_workers=5 es un número razonable para evitar sobrecargar el sitio web con demasiadas peticiones simultáneas
    with ThreadPoolExecutor(max_workers=5) as executor:
        # Enviamos cada categoría a la función que la procesa, y obtenemos una lista de resultados donde cada resultado es la lista de libros extraídos de esa categoría, lo que nos permite manejar el procesamiento de cada categoría de manera concurrente y luego unir los resultados al finalizar
        resultados = list(executor.map(lambda c: procesar_categoria(c, mi_sesion), categorias))
        
    # Unimos los resultados de todas las categorías en una sola lista de libros, iterando sobre cada sublista de libros que se obtuvo de cada categoría y extendiendo la lista principal de libros con esos datos, lo que nos da una colección completa de todos los libros extraídos de todas las categorías para su posterior procesamiento o análisis
    for sublista in resultados:
        lista_libros.extend(sublista)
        
    tiempo_fase1 = time.time() - inicio # Calculamos el tiempo que tardó la fase 1 restando el tiempo de inicio al tiempo actual, lo que nos da una medida del tiempo total que se invirtió en descargar los libros de todas las categorías y procesar esa información
    print(f"\n📦 FASE 1 COMPLETADA: {len(lista_libros)} libros descargados en {tiempo_fase1:.2f}s.")
    
    # --- FASE 2: ACTUALIZACIÓN DE AUTORES (Aquí se quita el "Pendiente") ---
    print("\n🕵️ FASE 2: Buscando autores para cada libro...")
    
    # Usamos ThreadPoolExecutor para actualizar los autores en paralelo
    # max_workers=4 es seguro para la API
    with ThreadPoolExecutor(max_workers=4) as executor:
        # Enviamos cada libro a la función que busca el autor
        # IMPORTANTE: Los diccionarios se modifican 'in-place' (se actualizan solos)
        futures = [executor.submit(obtener_autor, libro) for libro in lista_libros]
        
        # Esperamos a que terminen todos para asegurar que no quede ningún "Pendiente"
        total = len(futures)
        for i, future in enumerate(as_completed(futures), 1):
            if i % 50 == 0: # Imprimimos progreso cada 50 libros
                print(f"   ...Progreso Autores: {i}/{total} completados")

    print(f"\n🎉 ¡Misión Cumplida! Todos los datos están listos.")
    return lista_libros # Devolvemos la lista completa de libros con los datos actualizados, incluyendo el autor obtenido de la API de Google Books o el autor simulado en caso de no tener resultados, lo que nos da una colección final de todos los libros con su información completa para su uso posterior o análisis

# Punto de entrada del script
if __name__ == "__main__":
    lista_libros = ejecutar_scraping() # Llamamos a la función principal ejecutar_scraping para iniciar el proceso completo de scraping, lo que nos devuelve la lista de libros con toda la información extraída y actualizada, y almacenamos esa lista en la variable lista_libros para mostrar los resultados o realizar análisis adicionales
    print("\nDatos Libros")
    for i, libro in enumerate(lista_libros, 1):
        titulo = libro.get('titulo', 'N/A')
        autor = libro.get('autor', 'N/A')
        precio = libro.get('precio', 0.0)
        categoria = libro.get('categoria', 'N/A')
        rating = libro.get('rating', 0)
        stock = libro.get('stock', 'N/A')

        print(f"{i}- | {titulo} | {autor} | £{precio} | {categoria} | ⭐ {rating} | {stock}")

🚀 FASE 1: Descargando libros de todas las categorías...
✅ Categoría procesada: Travel........................ 11 libros
✅ Categoría procesada: Philosophy.................... 11 libros
✅ Categoría procesada: Classics...................... 19 libros
✅ Categoría procesada: Historical Fiction............ 26 libros
✅ Categoría procesada: Womens Fiction................ 17 libros
✅ Categoría procesada: Mystery....................... 32 libros
✅ Categoría procesada: Religion...................... 7 libros
✅ Categoría procesada: Romance....................... 35 libros
✅ Categoría procesada: Childrens..................... 29 libros
✅ Categoría procesada: Music......................... 13 libros
✅ Categoría procesada: Sequential Art................ 75 libros
✅ Categoría procesada: Science Fiction............... 16 libros
✅ Categoría procesada: Sports and Games.............. 5 libros
✅ Categoría procesada: Fiction....................... 65 libros
✅ Categoría procesada: New Adult..................

DML - DATA MANIPULATION LANGUAGE

In [ ]:
import sqlite3

# Funcion que guarda los datos de los libros en una base de datos SQLite.
def guardar_datos_en_db(lista_libros, libros_db="libros.db"):
    # Conectamos a la base de datos y obtenemos un cursor para ejecutar comandos SQL
    conexion = sqlite3.connect(libros_db)
    cursor = conexion.cursor()
    
    print(f"💾 Iniciando guardado de {len(lista_libros)} libros...")

    try:
        for libro in lista_libros:
            # --- A. INSERTAR CATEGORÍA ---
            # Usamos INSERT OR IGNORE para evitar duplicados
            cursor.execute("INSERT OR IGNORE INTO categorias (nombre) VALUES (?)", (libro['categoria'],))
            cursor.execute("SELECT id_categoria FROM categorias WHERE nombre = ?", (libro['categoria'],))
            cat_id = cursor.fetchone()[0] # Obtenemos el ID de la categoría recién insertada o existente para usarlo como clave foránea en la tabla de libros

            # --- B. INSERTAR AUTOR ---
            cursor.execute("INSERT OR IGNORE INTO autores (nombre_autor) VALUES (?)", (libro['autor'],))
            cursor.execute("SELECT id_autor FROM autores WHERE nombre_autor = ?", (libro['autor'],))
            autor_id = cursor.fetchone()[0] # Obtenemos el ID del autor recién insertado o existente para usarlo en la tabla intermedia de libro_autor, ya que un autor puede estar asociado a varios libros y viceversa

            # --- C. INSERTAR LIBRO ---
            cursor.execute('''
                INSERT INTO libros (titulo, precio, rating, stock, link, categoria_id) 
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (libro['titulo'], libro['precio'], libro['rating'], libro['stock'], libro['link'], cat_id)) # Insertamos el libro en la tabla de libros, incluyendo el título, precio, rating, stock, enlace y la clave foránea que referencia a la categoría a la que pertenece el libro
            
            libro_id = cursor.lastrowid # Obtenemos el ID del libro recién insertado para usarlo en la tabla intermedia de libro_autor

            # --- D. VINCULAR EN TABLA INTERMEDIA (Muchos a Muchos) ---
            cursor.execute('''
                INSERT OR IGNORE INTO libro_autor (libro_id, autor_id) 
                VALUES (?, ?)
            ''', (libro_id, autor_id)) # Insertamos la relación entre el libro y el autor en la tabla intermedia libro_autor

        # Confirmamos los cambios
        conexion.commit()
        print("✅ ¡Todos los datos se han guardado correctamente en la base de datos!")

    except Exception as e:
        conexion.rollback() # Si algo falla, deshacemos los cambios para no corromper la DB
        print(f"❌ Error al guardar en la DB: {e}")
    
    finally:
        conexion.close() # Cerramos la conexión a la base de datos para liberar recursos

# EJECUCIÓN: Llamamos a la función con la lista que se generó al ejecutar el scraping. Asegúrate de que la variable se llame igual (ej. lista_libros o lista_final)
guardar_datos_en_db(lista_libros)

💾 Iniciando guardado de 1000 libros...
✅ ¡Todos los datos se han guardado correctamente en la base de datos!


Conectar Base de datos!

In [ ]:
import sqlite3
import time
# Conectamos a la base de datos para verificar que los datos se guardaron correctamente
libro_db = 'libros.db'
conn = sqlite3.connect(libro_db)
cursor = conn.cursor()

Consulta 1.
"Libros con 5 estrellas ordenados por precio (Desc)"

In [ ]:
print("Consulta 1: Libro con 5 estrellas ordenadas por precio")

inicio = time.time()

cursor.execute ("""
    SELECT titulo, precio, rating
    FROM libros
    WHERE rating = 5
    ORDER BY precio DESC;
""") # Ejecutamos una consulta SQL para seleccionar el título, precio y rating de los libros que tienen un rating de 5 estrellas, ordenados por precio de forma descendente para mostrar primero los libros más caros con 5 estrellas
 
resultado = cursor.fetchall()

fin = time.time()

for row in resultado:
    print(row)

print(f"Tiempo de ejecucion: {fin - inicio:.4f} segundos")

Consulta 1: Libro con 5 estrellas ordenadas por precio
('The Barefoot Contessa Cookbook', 59.92, 5)
('Life Without a Recipe', 59.04, 5)
('Approval Junkie: Adventures in Caring Too Much', 58.81, 5)
('How to Speak Golf: An Illustrated Guide to Links Lingo', 58.32, 5)
('Digital Fortress', 58.0, 5)
('The Sound Of Love', 57.84, 5)
('Travels with Charley: In Search of America', 57.82, 5)
('El Deafo', 57.62, 5)
('H is for Hawk', 57.42, 5)
('Immunity: How Elie Metchnikoff Changed the Course of Modern Medicine', 57.36, 5)
('The Disappearing Spoon: And Other True Tales of Madness, Love, and the History of the World from the Periodic Table of the Elements', 57.35, 5)
('Kitchens of the Great Midwest', 57.2, 5)
('A Piece of Sky, a Grain of Rice: A Memoir in Four Meditations', 56.76, 5)
('Into the Wild', 56.7, 5)
('Eleanor & Park', 56.51, 5)
('Abstract City', 56.37, 5)
('The False Prince (The Ascendance Trilogy #1)', 56.0, 5)
('Future Shock (Future Shock #1)', 55.65, 5)
("A New Earth: Awakening to Y

CONSULTA 1 CON INDICE

In [8]:
inicio_indice = time.time()
cursor.execute("CREATE INDEX IF NOT EXISTS idx_libros_rating ON libros(rating)")
conn.commit()
fin_indice = time.time()

print(f"Tiempo en crear el indice: {fin_indice - inicio_indice:.4f} segundos")

print("Consulta 1: Libros con 5 estrellas ordenados por precio (DESC)")
inicio_consulta = time.time()

cursor.execute ("""
    SELECT titulo, precio, rating
    FROM libros
    WHERE rating = 5
    ORDER BY precio DESC;
""")

resultado = cursor.fetchall()

fin_consulta = time.time()

for row in resultado:
    print(row)

print(f"Tiempo de ejecucion de la consulta: {fin_consulta - inicio_consulta:.4f} segundos")

Tiempo en crear el indice: 0.0359 segundos
Consulta 1: Libros con 5 estrellas ordenados por precio (DESC)
('The Barefoot Contessa Cookbook', 59.92, 5)
('Life Without a Recipe', 59.04, 5)
('Approval Junkie: Adventures in Caring Too Much', 58.81, 5)
('How to Speak Golf: An Illustrated Guide to Links Lingo', 58.32, 5)
('Digital Fortress', 58.0, 5)
('The Sound Of Love', 57.84, 5)
('Travels with Charley: In Search of America', 57.82, 5)
('El Deafo', 57.62, 5)
('H is for Hawk', 57.42, 5)
('Immunity: How Elie Metchnikoff Changed the Course of Modern Medicine', 57.36, 5)
('The Disappearing Spoon: And Other True Tales of Madness, Love, and the History of the World from the Periodic Table of the Elements', 57.35, 5)
('Kitchens of the Great Midwest', 57.2, 5)
('A Piece of Sky, a Grain of Rice: A Memoir in Four Meditations', 56.76, 5)
('Into the Wild', 56.7, 5)
('Eleanor & Park', 56.51, 5)
('Abstract City', 56.37, 5)
('The False Prince (The Ascendance Trilogy #1)', 56.0, 5)
('Future Shock (Future 

Consulta 2: "Autores con mas de 5 libros."

In [ ]:
print("Consulta 2: Autores con mas de 1 libros")

cursor.execute("""
    SELECT 
        autores.nombre_autor, 
        COUNT(libro_autor.libro_id)
    FROM autores
    JOIN libro_autor ON autores.id_autor = libro_autor.autor_id
    GROUP BY autores.id_autor, autores.nombre_autor
    HAVING COUNT(libro_autor.libro_id) > 1
    ORDER BY COUNT(libro_autor.libro_id) DESC;
""")

resultado = cursor.fetchall()

for row in resultado:
    print(row)

Consulta 2: Autores con mas de 1 libros
('Autor Simulado', 846)
('Natsuki Takaya', 3)
('Kurtis J. Wiebe', 2)
('Lee Bermejo', 2)
('Neil Gaiman', 2)
('Sophie Kinsella', 2)
('Stephen King', 2)
('Dan Brown', 2)
('David Sedaris', 2)


Consulta 3: "Libros que cuestan 50 euros para arriba."

In [10]:
print("Consulta 3: Libros que cuestan 50 euros para arriba.")

cursor.execute("""
    SELECT titulo, precio
    FROM libros
    WHERE precio > 50.0
    ORDER BY precio DESC;
""")

for row in cursor.fetchall():
    print(row)

Consulta 3: Libros que cuestan 50 euros para arriba.
('The Perfect Play (Play by Play #1)', 59.99)
('Last One Home (New Beginnings #1)', 59.98)
('Civilization and Its Discontents', 59.95)
('The Barefoot Contessa Cookbook', 59.92)
('The Diary of a Young Girl', 59.9)
('The Bone Hunters (Lexy Vaughan & Steven Macaulay #2)', 59.71)
('Thomas Jefferson and the Tripoli Pirates: The Forgotten War That Changed American History', 59.64)
('Boar Island (Anna Pigeon #19)', 59.48)
('The Improbability of Love', 59.45)
('The Man Who Mistook His Wife for a Hat and Other Clinical Tales', 59.45)
('The Gray Rhino: How to Recognize and Act on the Obvious Dangers We Ignore', 59.15)
('Life Without a Recipe', 59.04)
('Listen to Me (Fusion #1)', 58.99)
('Unlimited Intuition Now', 58.87)
('Approval Junkie: Adventures in Caring Too Much', 58.81)
('Hamilton: The Revolution', 58.79)
('Myriad (Prentor #1)', 58.75)
('The Rose & the Dagger (The Wrath and the Dawn #2)', 58.64)
('Candide', 58.63)
('Alight (The Generati

Consulta 4: "Libros que tengan mas de 10 unidades en stock"

In [7]:
print("Consulta 4: Libros que tengan mas de 10 unidades en stock.")

cursor.execute("""
    SELECT titulo, stock
    FROM libros
    WHERE stock > 10
    ORDER BY stock DESC;
""")

for row in cursor.fetchall():
    print(row)

Consulta 4: Libros que tengan mas de 10 unidades en stock.
('A Light in the Attic', 22)
('Sharp Objects', 20)
('Tipping the Velvet', 20)
('Soumission', 20)
('Sapiens: A Brief History of Humankind', 20)
("It's Only the Himalayas", 19)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 19)
('Chase Me (Paris Nights #2)', 19)
('Black Dust', 19)
('Birdsong: A Story in Pictures', 19)
('Rip it Up and Start Again', 19)
('Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991', 19)
('How Music Works', 19)
('The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 19)
('The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 19)
('Starving Hearts (Triangular Trade Trilogy, #1)', 19)
("America's Cradle of Quarterbacks: Western Pennsylvania's Football Factory from Johnny Unitas to Joe Montana", 19)
('Aladdin and His Wonderful Lamp', 19)
('Mesaerion: The Best Science Fiction Stories 1800-1

CONSULTA 5: "Libros con más de 3 estrellas por menos de £10, para cuando estás en bancarrota pero con estándares"

In [ ]:
print("CONSULTA 5: Libros con más de 3 estrellas por menos de £15")

cursor.execute("""
    SELECT titulo, rating, precio
    FROM libros
    WHERE rating > 3 AND precio < 15.0
    ORDER BY rating DESC, precio ASC;
""")

for row in cursor.fetchall():
    print(row)

CONSULTA 5: Libros con más de 3 estrellas por menos de £10
('An Abundance of Katherines', 5, 10.0)
('Greek Mythic History', 5, 10.23)
('The Power Greens Cookbook: 140 Delicious Superfood Recipes', 5, 11.05)
('Dear Mr. Knightley', 5, 11.21)
('The Darkest Corners', 5, 11.33)
('Naturally Lean: 125 Nourishing Gluten-Free, Plant-Based Recipes--All Under 300 Calories', 5, 11.38)
('Fruits Basket, Vol. 2 (Fruits Basket #2)', 5, 11.64)
('Old School (Diary of a Wimpy Kid #10)', 5, 11.83)
('Superman Vol. 1: Before Truth (Superman by Gene Luen Yang #1)', 5, 11.89)
('Every Heart a Doorway (Every Heart A Doorway #1)', 5, 12.16)
('The Girl You Lost', 5, 12.29)
('The Silent Wife', 5, 12.34)
('Agnostic: A Spirited Manifesto', 5, 12.51)
('The Third Wave: An Entrepreneurâ\x80\x99s Vision of the Future', 5, 12.61)
("Walt Disney's Alice in Wonderland", 5, 12.96)
('Princess Between Worlds (Wide-Awake Princess #5)', 5, 13.34)
('Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Princess Jellyfish 2-in-1 Omnibus #1)

CIERRE!

In [13]:
# Si tu variable de conexión se llama 'conn'
try:
    conn.close()
    print("✅ Conexión cerrada.")
except NameError:
    print("⚠️ La variable 'conn' no existe o ya fue cerrada.")

✅ Conexión cerrada.
